# 06 実データに適用する — 合成データの結果は、どこまで通用するか

> 🔒 **このNotebookは自宅の発電量（個人データ）を読み込みます。** 日付や個別の値は表示せず、件数や傾向だけを出力します。
> データは `ml_forecast/data/` を**読み取り専用**で参照し、コピーしません。出力は消した状態でコミットします。
> 実データがない環境では、最初のセルでエラーになります（合成データの部分だけ試せます）。

## ゴール
- 合成データで作った検知器を、**実データ**に当てはめて、何が変わるかを確かめる。
- 合成データの結果が**実データでは通用しない**部分を理解する。
- 実データには正解がないので、**候補を出して人が確認する**運用の考え方を知る。

## 前のNotebookとのつながり
05 で、「日次の残差＋固着ルール」が合成データの異常を全種類見つけることを確認しました。
ただし合成データは単純な構造です。ここでは、日射量と発電量の関係が**どれだけ荒いか**を実データで測ります。

## やること
1. 実データの残差のばらつきを、合成データと比べる
2. 実データ並みにノイズを増やした合成データで、検知限界を調べる
3. 実データで、期待より大きく少ない日を候補として探す

## キーワード
- **残差のばらつき（MAD）**: 外れ値に強いばらつきの指標。これが大きいほど、小さな異常は見つけられない。
- **日射量のずれ**: 日射量は、自宅で測った値ではなく、気象メッシュデータ（観測から推定した格子の値）。自宅の天気と違う日は、期待値が外れる。
- **候補**: 異常と断定せず、「期待より大きく少ない」日として挙げたもの。原因（停電・影・出力制御・計測の欠け等）は人が確認する。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from real_data import load_real
import run_phase6 as p6

df = load_real()
print("日数:", len(df), " 列:", df.columns.tolist())
print("日射量と発電量の相関:", round(df["gsr"].corr(df["gen_kwh"]), 2),
      " 合成データ:", round(__import__("anomaly_synth").make_split("train")["gsr"].corr(
          __import__("anomaly_synth").make_split("train")["gen_kwh"]), 2))

**何をしているか**: 実データを読み込み、日射量と発電量の相関を、合成データと比べます。

**出力の見方**: 実データの相関は約0.86で、合成データ（ほぼ1.0）より**かなり弱い**です。
原因は、日射量が自宅で測った値ではなく、**メッシュ推定値**で、自宅の実際の日射とずれるためと考えられます（雲の位置、雨の降り方、季節ごとの発電効率の違いなど。本当の原因は未検証です）。

In [ ]:
r = p6.real_residuals(df)
cal = p6.synthetic_noise_calibration()
print("実データの相対残差のばらつき（MAD）:", round(p6.mad(r["rel"]), 3))
print(cal.to_string(index=False))
r.groupby("level", observed=True).agg(日数=("rel", "size"), 相対MAD=("rel", p6.mad), 絶対MAD_kWh=("short", p6.mad)).round(2)

**何をしているか**: 月ごとのhold-out（その月を学習に使わない）で、実データの残差のばらつきを測ります。下の表は、合成データの日射計測誤差を変えたときの値です。

**出力の見方**:
- 実データの相対残差のばらつきは約0.22。合成データの標準設定（日射計測誤差0.04）は約0.05なので、**約4〜5倍**です。
- 合成データで日射計測誤差を**0.20〜0.25**にすると、実データと同程度になります。つまり、実データは、日射量の計測が**5倍ほど荒い**状態に相当します。
- 期待発電量の水準ごとに見ると、15 kWh超の日の絶対残差は約2.3〜2.6 kWh、10〜15 kWhの日は約3.5 kWhです。**期待値の1〜3割がノイズ**です。

In [ ]:
noise = 0.20
shade = p6.noisy_shade(noise)
degrade = p6.noisy_degrade(noise)
print("影の強さと検知率（合成データ・日射計測誤差0.20）")
print(shade.to_string(index=False))
print("\n劣化の低下率ごとの警報率")
print(degrade.to_string(index=False))

**何をしているか**: 合成データの日射計測誤差を実データ並み（0.20）にして、05 と同じ感度分析をやり直します（検知器のしきい値も、ノイズの大きい学習データで決め直します）。

**出力の見方**:
- **影**: 05（標準ノイズ）では、20%低下で日単位1.0でした。実データ並みのノイズでは、**20%低下で日単位0.10、40%低下で0.51、50%で0.82、60%で0.96**です。**半分以上の低下でないと、確実には見つかりません**。
- **劣化**: 05 では6〜8%の低下から見つかり始めました。実データ並みのノイズでは、**20〜30%の低下でも日次0.06・7日平均0.25、30〜40%でようやく7日平均0.78**です。検知限界は、**約30%以上の低下**と言えます。
- つまり、合成データの「全種類をイベント単位で検知」は、実データの**荒さを考えると、強い異常に限った話**でした。
- なお、この見積もりは、「日射量の誤差が乗法的」と仮定した合成データでの目安で、実データのノイズの構造（雨の日に大きくぶれるなど）は完全には再現しません。

In [ ]:
cand = r[(r["z_abs"] > 3) | (r["z_rel"] > 3)]
print("候補（期待より大きく少ない日）:", len(cand), "日 / 全", len(r), "日")
print("  天気が雨:", int((cand["weather_class"] == 2).sum()), "日、晴れ側:", int((cand["weather_class"] == 0).sum()), "日")
print("  発電量が0の日（全体）:", int((df["gen_kwh"] == 0).sum()), "日")
print("  月別の件数:", cand["date"].dt.month.value_counts().sort_index().to_dict())

**何をしているか**: 実データで、期待より大きく少ない日（残差が、ばらつきの3倍超）を候補として数えます。日付などの詳細は、手元の `outputs/real/candidates.csv`（git管理外）にだけ書き出されます。

**出力の見方**:
- 候補は514日中15日（約3%）です。**大半（12日）が雨の日**で、期待発電量が小さい日に、実測が0近くまで落ちたものです。これは、約17か月で月0.9回の警報にあたり、設計した誤警報の目標（正常日の3%）と同じ水準です。
- **晴れ側の候補は3日**です。日射量は十分あるのに発電が期待を大きく下回っており、原因を確認する価値が高い日でした。**所有者に確認したところ、3日とも異常ではなく正常でした**（台風の接近1日、自宅付近で局地的に雨雲が発達した日2日）。メッシュの日射量は、こうした**自宅付近だけの天気**を反映できず、期待値が外れて**誤警報**になっています。
- **発電量がちょうど0の日が4日**あります。いずれも日射量が小さい雨の日で、所有者に確認したところ、**雨による正常な0**でした。
- **所有者に確認した結果、候補15日はすべて正常でした**（雨12日・台風接近1日・局地的な雨雲2日）。この実データでは、**候補の適合率は0/15**です。検知した異常はなく、「実データに検知すべき異常がなかった」のか「あっても見逃した」のかは、正解がないので判別できません。検知率も計算できません。候補は「異常」ではなく「確認を促す日」として扱い、確認の結果（正常・停電・影など）を記録すれば、将来の評価に使えます。
- この経験から、実データの誤警報は次の2つの型に分かれます。
  - **雨の日（12日）**: 期待発電量が小さい日（約2〜5 kWh）に、実測が0近くまで落ちる。相対残差・標準化した残差が大きく出るが、発電量が小さいので実害は小さい。期待発電量が小さい日（例: 5 kWh未満）を警報の対象外にする設計が考えられる。
  - **台風・局地的な雨雲（3日）**: 入力（日射量）が自宅付近だけの天気を捉えられない。天気が不安定な日は警報に注記する、自宅の日射計を使う、といった工夫が考えられる。

## 試してみよう（演習）
1. **しきい値を変える**: `cand` の条件 `> 3` を `> 2` に変えて、候補の数がどう変わるか見てください。分かるのは、しきい値を下げると候補は急増するが、大半は「ただの雨の日のぶれ」になる、ということです。
2. **月ごとのhold-outを年ごとに変える**: `src/run_phase6.py` の `real_residuals` で、`month = df["date"].dt.to_period("M")` を `df["date"].dt.to_period("Q")`（四半期）に変えて、MAD が変わるか比べてください。見るポイントは、MAD の増減です。学習から除く期間が長いと、季節の影響が残差に出てMADが増えるかもしれません（予想です。確かめてみましょう）。
3. **特徴を増やす**: `real_residuals` の `x = df[["gsr"]]` を `df[["gsr", "tmax"]]` に変えて、MAD が下がるか確かめてください。私が確かめたところ、気温を足してもMADはほとんど減りませんでした（0.227→0.224）。気温は主な原因ではなく、日射量の入力そのもののずれが効いていると考えられます。

## まとめ
- 実データの日射量はメッシュ推定値で、**合成データの標準設定より約5倍荒い**（残差のばらつきで比べた場合）。そのため、検知できる異常の大きさが大きく変わる。
- 合成データでは「影20%低下」「劣化6〜8%低下」から見つかったが、実データ並みのノイズでは、**影は40〜50%以上、劣化は30%以上**の低下が必要。
- 実データの結果は、**候補を出して人が確認する**形になる。確認の結果、候補15日は**すべて正常**（雨・台風接近・局地的な雨雲による低下）で、誤警報だった。正解がなければ、検知率は測れない。
- **合成データの結果をそのまま実データに当てはめない**。実データのノイズの大きさを先に測り、検知限界を見積もる。
- 実用に近づけるには、**自宅の日射計・パワコンの発電ログ**（時間単位）など、ノイズの小さい入力を使うのが効果的。